# Trayectorias por paradigma (Semana 3–4)

Lee JSON de `results/sweep/` o `results/smoke/`, arma series por paradigma y checkpoint, ajusta polinomio grado 5 (log-tokens) y clasifica topología (monótona / U / U invertida / oscilatoria). Ver `docs/04_experimental_design.md`.

Tras corridas, generar tabla larga con `python -m ontogenia aggregate --output-parquet ../results/aggregated_metrics.parquet` y cargar ese Parquet aquí como alternativa a parsear muchos JSON.

In [ ]:
from pathlib import Path

import pandas as pd

from ontogenia.topology import classify_all_tasks

ROOT = Path("..")
PARQUET = ROOT / "results" / "aggregated_metrics.parquet"
PARQUET.exists()

In [ ]:
if not PARQUET.exists():
    raise FileNotFoundError(
        "Falta aggregated_metrics.parquet. Ejecutar: "
        "python -m ontogenia aggregate --output-parquet results/aggregated_metrics.parquet"
    )

frame = pd.read_parquet(PARQUET)
# BLiMP tasks y métrica de accuracy principal del harness
traj = frame.dropna(subset=["training_step", "acc,none"]).copy()
traj = traj[traj["task"].astype(str).str.startswith("blimp")]

summary = classify_all_tasks(traj, metric_col="acc,none")
summary.sort_values(["model_size", "shape", "task"]).head(20)

In [ ]:
# Opcional: guardar clasificación topológica para paper/figuras
out = ROOT / "results" / "topology_summary.parquet"
summary.to_parquet(out, index=False)
out